In [43]:
from pathlib import Path
import numpy as np
import pandas as pd
print("pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

pandas version: 2.1.4
NumPy version: 1.26.4


In [44]:
PROJECT_ROOT = Path(
"../"
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Processed-data folder:", PROCESSED_DIR)
print("Folder exists:", PROCESSED_DIR.exists())

Project root: ..
Processed-data folder: ../data/processed
Folder exists: True


In [45]:
INPUT_FILE = PROCESSED_DIR / "student-mat-clean.csv"
print("Input file:", INPUT_FILE)
print("File exists:", INPUT_FILE.exists())

Input file: ../data/processed/student-mat-clean.csv
File exists: True


In [46]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find the input file:\n{INPUT_FILE}\n"
        "Check PROJECT_ROOT and the input filename."
    )

df = pd.read_csv(
    INPUT_FILE,
    sep=None,
    engine="python"
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

Dataset loaded successfully.
Dataset shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [47]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

Number of rows: 395
Number of columns: 33

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [48]:
dtype_table = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values
    }
)

display(dtype_table)

,column,dtype
0,school,object
1,sex,object
2,age,int64
3,address,object
4,famsize,object
5,Pstatus,object
6,Medu,int64
7,Fedu,int64
8,Mjob,object
9,Fjob,object


In [49]:
missing_summary = (
    df.isna()
        .sum()
        .sort_values(ascending=False)
        .to_frame("missing_count")
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

display(missing_summary.head(20))

,missing_count,missing_percent
school,0,0.0
paid,0,0.0
G2,0,0.0
G1,0,0.0
absences,0,0.0
health,0,0.0
Walc,0,0.0
Dalc,0,0.0
goout,0,0.0
freetime,0,0.0


In [50]:
print("Total missing values before encoding:", df.isna().sum().sum())

Total missing values before encoding: 0


In [51]:
cat_cols = df.select_dtypes(include="object").columns
print("Categorical columns:")
print(list(cat_cols))
print("\nNumber of categorical columns:", len(cat_cols))

Categorical columns:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of categorical columns: 17


In [52]:
if len(cat_cols) == 0:
    raise ValueError(
        "No object-type categorical columns were found. "
        "The dataset may already be encoded, or the categorical "
        "columns may use a different data type."
    )

print("Categorical columns were found. Encoding can continue.")

Categorical columns were found. Encoding can continue.


In [53]:
for column in cat_cols:
    unique_values = df[column].drop_duplicates().tolist()

    print("=" * 70)
    print("Column:", column)
    print("Number of nonmissing categories:", df[column].nunique(dropna=True))
    print("Number of missing values:", df[column].isna().sum())
    print("Example categories:", unique_values[:10])

Column: school
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GP', 'MS']
Column: sex
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['F', 'M']
Column: address
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['U', 'R']
Column: famsize
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GT3', 'LE3']
Column: Pstatus
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['A', 'T']
Column: Mjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['at_home', 'health', 'other', 'services', 'teacher']
Column: Fjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['teacher', 'other', 'services', 'health', 'at_home']
Column: reason
Number of nonmissing categories: 4
Number of missing values: 0
Example categories: ['course', 'other', 'home', 'reputation']
Column: g

In [54]:
possible_identifier_columns = []

for column in cat_cols:
    uniqueness_ratio = df[column].nunique(dropna=False) / len(df)

    if uniqueness_ratio > 0.90:
        possible_identifier_columns.append(
            {
                "column": column,
                "unique_values": df[column].nunique(dropna=False),
                "uniqueness_ratio": round(uniqueness_ratio, 3)
            }
        )

if possible_identifier_columns:
    print("Review these possible identifier columns before encoding:")
    display(pd.DataFrame(possible_identifier_columns))
else:
    print("No likely identifier columns were detected.")

No likely identifier columns were detected.


In [55]:
rows_before = df.shape[0]
columns_before = df.shape[1]

print("Rows before encoding:", rows_before)
print("Columns before encoding:", columns_before)

Rows before encoding: 395
Columns before encoding: 33


In [56]:
category_count_table = []

for column in cat_cols:
    category_count = df[column].nunique(dropna=True)
    dummy_count = max(category_count - 1, 0)

    category_count_table.append(
        {
            "original_column": column,
            "number_of_categories": category_count,
            "expected_dummy_columns": dummy_count
        }
    )

category_count_df = pd.DataFrame(category_count_table)

display(category_count_df)

,original_column,number_of_categories,expected_dummy_columns
0,school,2,1
1,sex,2,1
2,address,2,1
3,famsize,2,1
4,Pstatus,2,1
5,Mjob,5,4
6,Fjob,5,4
7,reason,4,3
8,guardian,3,2
9,schoolsup,2,1


In [57]:
number_of_numeric_columns = df.shape[1] - len(cat_cols)

expected_dummy_columns = (
    category_count_df["expected_dummy_columns"].sum()
)

expected_total_columns = (
    number_of_numeric_columns + expected_dummy_columns
)

print("Original numeric columns:", number_of_numeric_columns)
print("Expected dummy columns:", expected_dummy_columns)
print("Expected final column count:", expected_total_columns)

Original numeric columns: 16
Expected dummy columns: 26
Expected final column count: 42


In [58]:
cat_cols = df.select_dtypes(include="object").columns

df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True
)

print("Encoded shape:", df_encoded.shape)

Encoded shape: (395, 42)


In [59]:
rows_after = df_encoded.shape[0]
columns_after = df_encoded.shape[1]

print("Rows before encoding:", rows_before)
print("Rows after encoding:", rows_after)

print("\nColumns before encoding:", columns_before)
print("Columns after encoding:", columns_after)

print("\nNet change in columns:", columns_after - columns_before)

Rows before encoding: 395
Rows after encoding: 395

Columns before encoding: 33
Columns after encoding: 42

Net change in columns: 9


In [60]:
print("Expected final column count:", expected_total_columns)
print("Actual final column count:", columns_after)

if columns_after == expected_total_columns:
    print("Column-count check passed.")
else:
    print(
        "Column-count check requires review. "
        "Missing values or unusual category structures may affect the result."
    )

Expected final column count: 42
Actual final column count: 42
Column-count check passed.


In [61]:
original_columns = set(df.columns)
encoded_columns = set(df_encoded.columns)

removed_columns = [
    column for column in df.columns
    if column not in encoded_columns
]

new_columns = [
    column for column in df_encoded.columns
    if column not in original_columns
]

print("Original categorical columns removed:")
print(removed_columns)

print("\nNumber of new dummy columns:", len(new_columns))
print("\nFirst 30 new dummy columns:")
print(new_columns[:30])

Original categorical columns removed:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of new dummy columns: 26

First 30 new dummy columns:
['school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'schoolsup_yes', 'famsup_yes', 'paid_yes', 'activities_yes', 'nursery_yes', 'higher_yes', 'internet_yes', 'romantic_yes']


In [62]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,True,False,True,False,False,False,True,True,False,False
1,17,1,1,1,2,0,5,3,3,1,...,False,False,False,True,False,False,False,True,True,False
2,15,1,1,1,2,3,4,3,2,2,...,True,False,True,False,True,False,True,True,True,False
3,15,4,2,1,3,0,3,2,2,1,...,True,False,False,True,True,True,True,True,True,True
4,16,3,3,1,2,0,4,3,2,1,...,False,False,False,True,True,False,True,True,False,False


In [63]:
display(df_encoded[new_columns].head())

,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,False,False,True,False,False,False,False,False,False,False,...,True,False,True,False,False,False,True,True,False,False
1,False,False,True,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,True,False
2,False,False,True,True,True,False,False,False,False,False,...,True,False,True,False,True,False,True,True,True,False
3,False,False,True,False,True,True,False,False,False,False,...,True,False,False,True,True,True,True,True,True,True
4,False,False,True,False,True,False,True,False,False,False,...,False,False,False,True,True,False,True,True,False,False


In [64]:
boolean_columns = df_encoded.select_dtypes(include="bool").columns
print("Number of Boolean dummy columns:", len(boolean_columns))

Number of Boolean dummy columns: 26


In [65]:
df_encoded[boolean_columns] = (
    df_encoded[boolean_columns]
    .astype("int8")
)

print("Boolean dummy variables converted to 0 and 1.")

Boolean dummy variables converted to 0 and 1.


In [66]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [67]:
remaining_object_columns = (
    df_encoded
    .select_dtypes(include="object")
    .columns
    .tolist()
)

print("Remaining object columns:", remaining_object_columns)

Remaining object columns: []


In [68]:
assert len(remaining_object_columns) == 0, (
    "Some object-type columns remain after encoding."
)

print("Object-column check passed.")

Object-column check passed.


In [69]:
assert df_encoded.shape[0] == df.shape[0], (
    "The number of rows changed during encoding."
)

print("Row-count check passed.")

Row-count check passed.


In [70]:
assert df_encoded.index.equals(df.index), (
    "The row index changed during encoding."
)

print("Row-index check passed.")

Row-index check passed.


In [71]:
if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "The G3 target variable is missing after encoding."
    )

    pd.testing.assert_series_equal(
        df_encoded["G3"],
        df["G3"],
        check_names=True
    )

    print("G3 target-variable check passed.")
else:
    print("G3 was not found in the current dataset.")

G3 target-variable check passed.


In [72]:
duplicate_column_count = (
    df_encoded.columns.duplicated().sum()
)

print("Duplicate column names:", duplicate_column_count)

assert duplicate_column_count == 0, (
    "Duplicate column names were detected."
)

print("Duplicate-column check passed.")

Duplicate column names: 0
Duplicate-column check passed.


In [73]:
missing_after = df_encoded.isna().sum().sum()
print("Total missing values before encoding:", df.isna().sum().sum())
print("Total missing values after encoding:", missing_after)

Total missing values before encoding: 0
Total missing values after encoding: 0


In [74]:
columns_with_missing_values = (
    df_encoded.isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
)

if columns_with_missing_values.empty:
    print("No missing values remain.")
else:
    print("Columns still containing missing values:")
    display(columns_with_missing_values.to_frame("missing_count"))

No missing values remain.


In [75]:
assert df_encoded.shape[0] == df.shape[0], (
    "Sanity check failed: row count changed."
)
assert df_encoded.index.equals(df.index), (
    "Sanity check failed: row index changed."
)

assert not df_encoded.columns.duplicated().any(), (
    "Sanity check failed: duplicate column names exist."
)

remaining_objects = (
    df_encoded.select_dtypes(include="object").columns.tolist()
)

assert len(remaining_objects) == 0, (
    f"Sanity check failed: object columns remain: {remaining_objects}"
)

if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "Sanity check failed: G3 is missing."
    )

    assert df_encoded["G3"].equals(df["G3"]), (
        "Sanity check failed: G3 values changed."
    )

print("All one-hot encoding sanity checks passed.")

All one-hot encoding sanity checks passed.


In [76]:
encoding_summary = pd.DataFrame(
    {
        "measurement": [
            "Rows before encoding",
            "Rows after encoding",
            "Columns before encoding",
            "Columns after encoding",
            "Categorical columns encoded",
            "New dummy columns",
            "Remaining object columns",
            "Missing values after encoding"
        ],
        "value": [
            df.shape[0],
            df_encoded.shape[0],
            df.shape[1],
            df_encoded.shape[1],
            len(cat_cols),
            len(new_columns),
            len(remaining_object_columns),
            df_encoded.isna().sum().sum()
        ]
    }
)

display(encoding_summary)

,measurement,value
0,Rows before encoding,395
1,Rows after encoding,395
2,Columns before encoding,33
3,Columns after encoding,42
4,Categorical columns encoded,17
5,New dummy columns,26
6,Remaining object columns,0
7,Missing values after encoding,0


In [77]:
OUTPUT_FILE = PROCESSED_DIR / "student-mat-encoded.csv"

df_encoded.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Encoded dataset saved successfully.")
print("Output file:", OUTPUT_FILE)

Encoded dataset saved successfully.
Output file: ../data/processed/student-mat-encoded.csv


In [78]:
assert OUTPUT_FILE.exists(), (
    f"The encoded output file was not created: {OUTPUT_FILE}"
)

saved_size = OUTPUT_FILE.stat().st_size

print("Saved file exists:", OUTPUT_FILE.exists())
print("Saved file size:", saved_size, "bytes")

Saved file exists: True
Saved file size: 34846 bytes


In [79]:
df_encoded_check = pd.read_csv(OUTPUT_FILE)

print("Original encoded shape:", df_encoded.shape)
print("Reloaded encoded shape:", df_encoded_check.shape)

assert df_encoded_check.shape == df_encoded.shape, (
    "The reloaded dataset shape does not match."
)

print("Saved-file verification passed.")

Original encoded shape: (395, 42)
Reloaded encoded shape: (395, 42)
Saved-file verification passed.


# One-Hot Encoding Review

## Summary

Your one-hot encoding step appears to have worked correctly based on the information provided.

- **Original dataset shape:** `(395, 33)`
- **Encoded dataset shape:** `(395, 42)`
- **Rows after encoding:** **395** (unchanged, which is expected)
- **Object columns remaining:** **None** (all categorical variables were encoded)
- **New dummy columns created:** **26**

The code used is a standard and appropriate way to one-hot encode categorical variables for many machine learning models.

```python
cat_cols = df.select_dtypes(include="object").columns

df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True
)

print("Encoded shape:", df_encoded.shape)
```

---

## Interpretation

### 1. Did the categorical columns expand correctly?

Yes.

The original dataset contained **17 categorical (`object`) columns**:

- `school`
- `sex`
- `address`
- `famsize`
- `Pstatus`
- `Mjob`
- `Fjob`
- `reason`
- `guardian`
- `schoolsup`
- `famsup`
- `paid`
- `activities`
- `nursery`
- `higher`
- `internet`
- `romantic`

After running `pd.get_dummies()`:

- All object columns were replaced with numeric dummy variables.
- No object columns remain.
- New binary (0/1) columns were created.

This is the expected result of one-hot encoding.

---

### 2. Why did the number of columns increase?

Each categorical variable is converted into one or more binary (0/1) columns.

For example, the original column:

| school |
|--------|
| GP |
| MS |

becomes:

| school_MS |
|-----------|
| 0 |
| 1 |

Variables with more than two categories produce multiple dummy columns.

For example:

| reason |
|---------|
| home |
| course |
| reputation |
| other |

becomes:

| reason_course | reason_home | reason_reputation |
|----------------|-------------|-------------------|
| 0/1 | 0/1 | 0/1 |

The omitted category (`other`) is represented whenever all three dummy variables equal **0**.

Because one original column can become several new columns, the total number of columns increases.

---

### 3. What is the purpose of `drop_first=True`?

The parameter `drop_first=True` removes one dummy column for each categorical variable.

Without `drop_first=True`:

- A variable with **k** categories creates **k** dummy columns.

With `drop_first=True`:

- A variable with **k** categories creates **k − 1** dummy columns.

This prevents **perfect multicollinearity**, also known as the **dummy variable trap**, where one dummy variable can be perfectly predicted from the others.

This is especially important for models such as:

- Linear Regression
- Logistic Regression

Although tree-based models (Decision Trees, Random Forests, Gradient Boosting) are less affected, using `drop_first=True` is still common practice.

---

### 4. How does the omitted category become the reference category?

The category that is dropped becomes the **reference (baseline) category**.

For example, suppose the `guardian` column contains:

- father
- mother
- other

After encoding with `drop_first=True`, the dataset may contain:

- `guardian_mother`
- `guardian_other`

If a student's encoded values are:

| guardian_mother | guardian_other |
|-----------------|----------------|
| 0 | 0 |

then the student's guardian is:

```
father
```

The remaining dummy variables are interpreted relative to this baseline.

For example:

- `guardian_mother = 1` means the guardian is **mother** instead of **father**.
- `guardian_other = 1` means the guardian is **other** instead of **father**.

The omitted category is therefore still represented—it is encoded implicitly.

---

### 5. Suggested sanity checks

After encoding, perform a few quick checks to confirm everything worked correctly.

#### Check for remaining object columns

```python
print(df_encoded.select_dtypes(include="object").columns)
```

Expected output:

```python
Index([], dtype='object')
```

---

#### Verify the dataset shape

```python
print(df_encoded.shape)
```

The output should confirm that the dataset still contains **395 rows**.

---

#### Inspect the new column names

```python
print(df_encoded.columns)
```

You should see new columns similar to:

- `school_MS`
- `sex_M`
- `address_U`
- `guardian_mother`
- `reason_reputation`

---

#### Check for missing values

```python
print(df_encoded.isnull().sum().sum())
```

Ideally, the result should be:

```
0
```

unless the original dataset already contained missing values.

---

### 6. Should the number of rows remain unchanged?

Yes.

One-hot encoding changes the **columns**, not the **rows**.

Each student remains one observation, so the dataset should still contain:

- **395 rows before encoding**
- **395 rows after encoding**

If the number of rows changes, another preprocessing step—not one-hot encoding—is responsible.

---

### 7. Possible risks

Although the encoding appears correct, there are several potential issues to consider.

#### Missing categories in future data

New data may contain categories that were not present during training.

For example:

Training data:

```
reason
```

contains:

- home
- course
- reputation

Later, new data contains:

```
reason = internship
```

The model will not know how to encode this unseen category unless the preprocessing pipeline is designed to handle unknown values.

---

#### Identifier columns

Identifier columns (such as Student ID or Record Number) should generally **not** be one-hot encoded because they do not provide useful predictive information.

The Student Performance dataset does not contain identifier columns, so this is not currently an issue.

---

#### High-cardinality variables

Variables with many unique categories (such as hundreds of cities or names) can generate hundreds of dummy variables.

This can:

- increase memory usage,
- slow model training,
- increase the risk of overfitting.

The Student Performance dataset contains only a small number of categories per variable, so this is not a concern.

---

#### Inconsistent category labels

One-hot encoding treats different spellings as different categories.

For example:

- `GP`
- `gp`
- ` GP`

would all become separate dummy variables.

Before encoding, categorical data should be cleaned by removing extra spaces and ensuring consistent capitalization.

---

## Recommendation

The one-hot encoding process appears to have been completed successfully.

Key observations:

- ✅ All categorical variables were converted into numeric dummy variables.
- ✅ The number of rows remained unchanged (**395**), which is expected.
- ✅ The increase from **33** to **42** columns is normal because categorical variables expand into multiple binary columns.
- ✅ Using `drop_first=True` avoids redundant dummy variables and establishes a reference category for each encoded feature.
- ✅ No object-type columns remain, indicating that the dataset is ready for most machine learning algorithms.

Before training your models, it is recommended to:

1. Verify that there are no missing values after preprocessing.
2. Review the new dummy variable names to ensure they match the expected categories.
3. Apply the same encoding process to future or test datasets so that the feature columns remain consistent.
4. Be prepared to handle unseen categories in future data by using the same fitted preprocessing pipeline during model deployment.

# GSSRP 2026 — Session 20
## Feature Engineering II: Full-Information Feature Set
This notebook creates the full-information modeling dataset. The predictors include
G1 and G2, while G3 is separated as the target variable.

In [80]:
if "df_encoded" in globals():
    print("df_encoded is already loaded.")
    print("Shape:", df_encoded.shape)
else:
    print("df_encoded is not currently loaded.")
    print("Load the encoded dataset created during Session 19.")

df_encoded is already loaded.
Shape: (395, 42)


In [81]:
if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"The processed-data directory was not found: {PROCESSED_DIR}"
    )

processed_files = sorted(PROCESSED_DIR.iterdir())

print("Files in data/processed/:")
for file_path in processed_files:
    print("-", file_path.name)

Files in data/processed/:
- student-mat-clean.csv
- student-mat-encoded.csv


In [82]:
ENCODED_FILE = PROCESSED_DIR / "student-mat-encoded.csv"
if not ENCODED_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {ENCODED_FILE}\n"
        "Replace the filename with the encoded dataset created in Session 19."
)

df_encoded = pd.read_csv(ENCODED_FILE)

print("Encoded dataset loaded successfully.")
print("Shape:", df_encoded.shape)

Encoded dataset loaded successfully.
Shape: (395, 42)


In [83]:
print("Encoded DataFrame shape:", df_encoded.shape)
print("\nFirst five rows:")
display(df_encoded.head())

Encoded DataFrame shape: (395, 42)

First five rows:


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [84]:
print("Number of columns:", len(df_encoded.columns))

print("\nColumn names:")
for column_number, column_name in enumerate(df_encoded.columns, start=1):
    print(f"{column_number}. {column_name}")

Number of columns: 42

Column names:
1. age
2. Medu
3. Fedu
4. traveltime
5. studytime
6. failures
7. famrel
8. freetime
9. goout
10. Dalc
11. Walc
12. health
13. absences
14. G1
15. G2
16. G3
17. school_MS
18. sex_M
19. address_U
20. famsize_LE3
21. Pstatus_T
22. Mjob_health
23. Mjob_other
24. Mjob_services
25. Mjob_teacher
26. Fjob_health
27. Fjob_other
28. Fjob_services
29. Fjob_teacher
30. reason_home
31. reason_other
32. reason_reputation
33. guardian_mother
34. guardian_other
35. schoolsup_yes
36. famsup_yes
37. paid_yes
38. activities_yes
39. nursery_yes
40. higher_yes
41. internet_yes
42. romantic_yes


In [85]:
required_grade_columns = ["G1", "G2", "G3"]
missing_grade_columns = [
    column
    for column in required_grade_columns
    if column not in df_encoded.columns
]

if missing_grade_columns:
    raise KeyError(
        "The following required columns are missing: "
        + ", ".join(missing_grade_columns)
    )
    
print("Verification passed: G1, G2, and G3 are present.")

Verification passed: G1, G2, and G3 are present.


In [86]:
duplicate_columns = df_encoded.columns[
    df_encoded.columns.duplicated()
].tolist()

if duplicate_columns:
    raise ValueError(
        "Duplicate column names were found: "
        + ", ".join(duplicate_columns)
    )
    
print("Verification passed: no duplicate column names were found.")

Verification passed: no duplicate column names were found.


In [87]:
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

print("Full-information features:", X_full.shape[1])

Full-information features: 41


In [88]:
print("Original encoded dataset shape:", df_encoded.shape)
print("Feature matrix shape:", X_full.shape)
print("Target vector shape:", y.shape)

Original encoded dataset shape: (395, 42)
Feature matrix shape: (395, 41)
Target vector shape: (395,)


In [89]:
assert "G3" not in X_full.columns, (
    "Target leakage detected: G3 is still present in X_full."
)
print("Verification passed: G3 is not present in X_full.")

Verification passed: G3 is not present in X_full.


In [90]:
assert "G1" in X_full.columns, (
    "G1 is missing from the full-information feature set."
)
assert "G2" in X_full.columns, (
    "G2 is missing from the full-information feature set."
)
print("Verification passed: G1 and G2 are included in X_full.")

Verification passed: G1 and G2 are included in X_full.


In [91]:
assert len(X_full) == len(y), (
    "The feature matrix and target have different row counts."
)
assert X_full.index.equals(y.index), (
    "The feature matrix and target indexes do not match."
)
print("Verification passed: X_full and y have matching rows and indexes.")

Verification passed: X_full and y have matching rows and indexes.


In [92]:
expected_feature_count = df_encoded.shape[1] - 1
actual_feature_count = X_full.shape[1]
assert actual_feature_count == expected_feature_count, (
    f"Expected {expected_feature_count} features, "
    f"but found {actual_feature_count}."
)
print("Expected feature count:", expected_feature_count)
print("Actual feature count:", actual_feature_count)
print("Feature-count verification passed.")

Expected feature count: 41
Actual feature count: 41
Feature-count verification passed.


In [93]:
print("First five rows of X_full:")
display(X_full.head())

First five rows of X_full:


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [94]:
print("First 20 feature names:")
for column_number, column_name in enumerate(
    X_full.columns[:20],
    start=1
):
    print(f"{column_number}. {column_name}")

First 20 feature names:
1. age
2. Medu
3. Fedu
4. traveltime
5. studytime
6. failures
7. famrel
8. freetime
9. goout
10. Dalc
11. Walc
12. health
13. absences
14. G1
15. G2
16. school_MS
17. sex_M
18. address_U
19. famsize_LE3
20. Pstatus_T


In [95]:
print("Target name:", y.name)
print("Target data type:", y.dtype)
print("Number of target observations:", len(y))
print("\nFirst five target values:")
display(y.head().to_frame())

Target name: G3
Target data type: int64
Number of target observations: 395

First five target values:


,G3
0,6
1,6
2,10
3,15
4,10


In [96]:
print("G3 descriptive statistics:")
display(y.describe())

G3 descriptive statistics:


count    395.000000
mean      10.415190
std        4.581443
min        0.000000
25%        8.000000
50%       11.000000
75%       14.000000
max       20.000000
Name: G3, dtype: float64

In [97]:
print("G3 frequency table:")
display(
    y.value_counts()
    .sort_index()
    .rename_axis("G3")
    .reset_index(name="Count")
)

G3 frequency table:


,G3,Count
0,0,38
1,4,1
2,5,7
3,6,15
4,7,9
5,8,32
6,9,28
7,10,56
8,11,47
9,12,31


In [98]:
feature_missing_values = int(X_full.isna().sum().sum())
target_missing_values = int(y.isna().sum())
print("Missing values in X_full:", feature_missing_values)
print("Missing values in y:", target_missing_values)

Missing values in X_full: 0
Missing values in y: 0


In [99]:
feature_type_counts = (
    X_full.dtypes
    .astype(str)
    .value_counts()
    .rename_axis("Data_Type")
    .reset_index(name="Column_Count")
)
display(feature_type_counts)

,Data_Type,Column_Count
0,int64,41


In [100]:
non_numeric_columns = X_full.select_dtypes(
    exclude="number"
).columns.tolist()

if non_numeric_columns:
    print("Nonnumeric feature columns were found:")
    for column in non_numeric_columns:
        print("-", column)
else:
    print("Verification passed: all features are numeric.")

Verification passed: all features are numeric.


In [101]:
print("=" * 60)
print("SESSION 20 FULL-INFORMATION DATASET SUMMARY")
print("=" * 60)
print(f"Number of observations: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"Target variable: {y.name}")
print(f"G1 included: {'G1' in X_full.columns}")
print(f"G2 included: {'G2' in X_full.columns}")
print(f"G3 excluded from features: {'G3' not in X_full.columns}")
print(f"Feature missing values: {X_full.isna().sum().sum()}")
print(f"Target missing values: {y.isna().sum()}")
print("=" * 60)
print("The full-information feature set is ready.")
print("=" * 60)

SESSION 20 FULL-INFORMATION DATASET SUMMARY
Number of observations: 395
Number of features: 41
Target variable: G3
G1 included: True
G2 included: True
G3 excluded from features: True
Feature missing values: 0
Target missing values: 0
The full-information feature set is ready.


In [103]:
feature_category_definitions = {
    "School and demographics": [
        "school",
        "sex",
        "age",
        "address"
    ],
    "Family background": [
        "famsize",
        "Pstatus",
        "Medu",
        "Fedu",
        "Mjob",
        "Fjob",
        "guardian",
        "famrel"
    ],
    "Academic history and school choices": [
        "reason",
        "traveltime",
        "studytime",
        "failures",
        "schoolsup",
        "paid",
        "activities",
        "nursery",
        "higher",
        "G1",
        "G2"
    ],
    "Family and educational support": [
        "famsup",
        "internet"
    ],
    "Lifestyle and social behavior": [
        "romantic",
        "freetime",
        "goout",
        "Dalc",
        "Walc"
    ],
    "Health and attendance": [
        "health",
        "absences"
    ]
}

In [104]:
def feature_matches_original(encoded_feature, original_feature):
    """
    Return True when an encoded feature corresponds to an original feature.
    Examples:
    - G1 matches G1
    - age matches age
    - Mjob_teacher matches Mjob
    - reason_home matches reason
    """
    return (
        encoded_feature == original_feature
        or encoded_feature.startswith(original_feature + "_")
    )

In [107]:
grouped_features = {
    category: []
    for category in feature_category_definitions
}

assigned_features = set()

for category, original_features in feature_category_definitions.items():
    for encoded_feature in X_full.columns:
        for original_feature in original_features:
            if feature_matches_original(
                encoded_feature,
                original_feature
            ):
                grouped_features[category].append(encoded_feature)
                assigned_features.add(encoded_feature)
                break

In [108]:
ungrouped_features = [
    feature
    for feature in X_full.columns
    if feature not in assigned_features
]
print("Number of grouped features:", len(assigned_features))
print("Number of ungrouped features:", len(ungrouped_features))

Number of grouped features: 41
Number of ungrouped features: 0


In [109]:
for category, features in grouped_features.items():
    print("\n" + "=" * 70)
    print(category.upper())
    print("=" * 70)
    print("Feature count:", len(features))
    if features:
        for feature_number, feature_name in enumerate(
            features,
            start=1
        ):
            print(f"{feature_number}. {feature_name}")
    else:
        print("No features were assigned to this category.")


SCHOOL AND DEMOGRAPHICS
Feature count: 4
1. age
2. school_MS
3. sex_M
4. address_U

FAMILY BACKGROUND
Feature count: 15
1. Medu
2. Fedu
3. famrel
4. famsize_LE3
5. Pstatus_T
6. Mjob_health
7. Mjob_other
8. Mjob_services
9. Mjob_teacher
10. Fjob_health
11. Fjob_other
12. Fjob_services
13. Fjob_teacher
14. guardian_mother
15. guardian_other

ACADEMIC HISTORY AND SCHOOL CHOICES
Feature count: 13
1. traveltime
2. studytime
3. failures
4. G1
5. G2
6. reason_home
7. reason_other
8. reason_reputation
9. schoolsup_yes
10. paid_yes
11. activities_yes
12. nursery_yes
13. higher_yes

FAMILY AND EDUCATIONAL SUPPORT
Feature count: 2
1. famsup_yes
2. internet_yes

LIFESTYLE AND SOCIAL BEHAVIOR
Feature count: 5
1. freetime
2. goout
3. Dalc
4. Walc
5. romantic_yes

HEALTH AND ATTENDANCE
Feature count: 2
1. health
2. absences


In [110]:
print("\n" + "=" * 70)
print("UNGROUPED FEATURES")
print("=" * 70)
if ungrouped_features:
    for feature_number, feature_name in enumerate(
        ungrouped_features,
        start=1
    ):
        print(f"{feature_number}. {feature_name}")
else:
    print("All features were assigned to a category.")


UNGROUPED FEATURES
All features were assigned to a category.


In [111]:
import pandas as pd
category_summary = pd.DataFrame({
    "Category": list(grouped_features.keys()),
    "Feature_Count": [
        len(features)
        for features in grouped_features.values()
    ]
})

category_summary.loc[len(category_summary)] = {
    "Category": "Ungrouped",
    "Feature_Count": len(ungrouped_features)
}
display(category_summary)

,Category,Feature_Count
0,School and demographics,4
1,Family background,15
2,Academic history and school choices,13
3,Family and educational support,2
4,Lifestyle and social behavior,5
5,Health and attendance,2
6,Ungrouped,0


In [112]:
feature_category_rows = []
for category, features in grouped_features.items():
    for feature in features:
        feature_category_rows.append({
            "Feature": feature,
            "Category": category
        })
for feature in ungrouped_features:
    feature_category_rows.append({
        "Feature": feature,
        "Category": "Ungrouped"
    })
feature_category_table = pd.DataFrame(
    feature_category_rows
).sort_values(
    by=["Category", "Feature"]
).reset_index(drop=True)
display(feature_category_table)

,Feature,Category
0,G1,Academic history and school choices
1,G2,Academic history and school choices
2,activities_yes,Academic history and school choices
3,failures,Academic history and school choices
4,higher_yes,Academic history and school choices
5,nursery_yes,Academic history and school choices
6,paid_yes,Academic history and school choices
7,reason_home,Academic history and school choices
8,reason_other,Academic history and school choices
9,reason_reputation,Academic history and school choices


In [113]:
def find_original_feature(encoded_feature):
    for category, original_features in feature_category_definitions.items():
        for original_feature in original_features:
           if feature_matches_original(
                encoded_feature,
                original_feature
           ):
                return original_feature
    return "Unidentified"

feature_category_table["Original_Feature"] = (
    feature_category_table["Feature"]
    .apply(find_original_feature)
)
feature_category_table = feature_category_table[
    ["Feature", "Original_Feature", "Category"]
]
display(feature_category_table)

,Feature,Original_Feature,Category
0,G1,G1,Academic history and school choices
1,G2,G2,Academic history and school choices
2,activities_yes,activities,Academic history and school choices
3,failures,failures,Academic history and school choices
4,higher_yes,higher,Academic history and school choices
5,nursery_yes,nursery,Academic history and school choices
6,paid_yes,paid,Academic history and school choices
7,reason_home,reason,Academic history and school choices
8,reason_other,reason,Academic history and school choices
9,reason_reputation,reason,Academic history and school choices


In [114]:
X_full_save = X_full.reset_index(drop=True).copy()
y_save = y.reset_index(drop=True).copy()
print("Prepared X_full shape:", X_full_save.shape)
print("Prepared y shape:", y_save.shape)

Prepared X_full shape: (395, 41)
Prepared y shape: (395,)


In [120]:
y_full_df = y_save.to_frame(name="G3")
display(y_full_df.head())

,G3
0,6
1,6
2,10
3,15
4,10


In [118]:
X_FULL_CSV = PROCESSED_DIR / "X_full.csv"
Y_FULL_CSV = PROCESSED_DIR / "y_full.csv"
print("X_full file:", X_FULL_CSV)
print("y file:", Y_FULL_CSV)

X_full file: ../data/processed/X_full.csv
y file: ../data/processed/y_full.csv


In [121]:
X_full_save.to_csv(
    X_FULL_CSV,
    index=False
)
y_full_df.to_csv(
    Y_FULL_CSV,
    index=False
)
print("Separate CSV files saved successfully.")
print("Saved:", X_FULL_CSV.name)
print("Saved:", Y_FULL_CSV.name)

Separate CSV files saved successfully.
Saved: X_full.csv
Saved: y_full.csv


# Session 21: Early-Warning Dataset Design
## Section 3: Create the Early-Warning Feature Matrix
The early-warning feature matrix excludes G1 and G2 so that the model can
make predictions before students' first-period and second-period grades are
available.

In [122]:
X_FULL_CSV = (
    PROCESSED_DIR / "X_full.csv"
)
Y_FULL_CSV = (
    PROCESSED_DIR / "y_full.csv"
)
print("Session 20 files:")
print(
    "X_full.csv exists:",
    X_FULL_CSV.exists()
)
print(
    "y_full.csv exists:",
    Y_FULL_CSV.exists()
)

Session 20 files:
X_full.csv exists: True
y_full.csv exists: True


In [126]:
X_full = pd.read_csv(
    X_FULL_CSV
)
y_loaded = pd.read_csv(
    Y_FULL_CSV
)
file_format_used = "CSV"

# Extract G3 as a pandas Series.
if isinstance(y_loaded, pd.DataFrame):
    if "G3" not in y_loaded.columns:
        raise KeyError(
            "The target file does not contain a G3 column."
        )
    y = y_loaded["G3"].copy()
else:
    y = y_loaded.copy()
# Reset indexes to ensure row alignment.
X_full = (
    X_full
    .reset_index(drop=True)
    .copy()
)
y = (
    y
    .reset_index(drop=True)
    .copy()
)
y.name = "G3"
print(
    "Files loaded successfully using:",
    file_format_used
)
print(
    "X_full shape:",
    X_full.shape
)
print(
    "y shape:",
    y.shape
)

Files loaded successfully using: CSV
X_full shape: (395, 41)
y shape: (395,)


In [128]:
print("First five rows of X_full:")
display(
    X_full.head()
)
print("\nFirst five target values:")
display(
    y.head().to_frame()
)

First five rows of X_full:


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0



First five target values:


,G3
0,6
1,6
2,10
3,15
4,10


In [131]:
assert isinstance(
    X_full,
    pd.DataFrame
), "X_full must be a pandas DataFrame."
assert isinstance(
    y,
    pd.Series
), "y must be a pandas Series."
assert "G3" not in X_full.columns, (
    "Target leakage detected: "
    "G3 is present in X_full."
)
assert "G1" in X_full.columns, (
    "G1 is missing from X_full."
)
assert "G2" in X_full.columns, (
    "G2 is missing from X_full."
)
assert y.name == "G3", (
    f"The target should be named G3, "
    f"but it is named {y.name}."
)
assert len(X_full) == len(y), (
    "X_full and y have different row counts."
)
assert X_full.index.equals(y.index), (
    "X_full and y do not have matching indexes."
)
print(
    "Session 20 input validation passed."
)
print(
    "G3 excluded from X_full:",
    "G3" not in X_full.columns
)
print(
    "G1 present in X_full:",
    "G1" in X_full.columns
)
print(
    "G2 present in X_full:",
    "G2" in X_full.columns
)
print(
    "Target stored in y:",
    y.name
)
print(
    "Matching row counts:",
    len(X_full) == len(y)
)

Session 20 input validation passed.
G3 excluded from X_full: True
G1 present in X_full: True
G2 present in X_full: True
Target stored in y: G3
Matching row counts: True


In [132]:
full_feature_count = X_full.shape[1]
print(
    "Full-information features:",
    full_feature_count
)

Full-information features: 41


In [133]:
leak_cols = [
    column
    for column in X_full.columns
    if column in ("G1", "G2")
]
print(
    "Columns identified for removal:",
    leak_cols
)

Columns identified for removal: ['G1', 'G2']


In [134]:
expected_removed_columns = {
    "G1",
    "G2"
}
actual_removed_columns = set(
    leak_cols
)
if (
    actual_removed_columns
    != expected_removed_columns
):
    raise ValueError(
        "The early-warning dataset must remove "
        "both G1 and G2. "
        f"Columns found: {leak_cols}"
    )
print(
    "Both G1 and G2 were found."
)

Both G1 and G2 were found.


In [135]:
X_early = X_full.drop(
    columns=leak_cols
).copy()
print(
    "Early-warning features:",
    X_early.shape[1]
)

Early-warning features: 39


In [136]:
print(
    "X_full shape:",
    X_full.shape
)
print(
    "X_early shape:",
    X_early.shape
)
print(
    "y shape:",
    y.shape
)
print(
    "Number of removed features:",
    X_full.shape[1] - X_early.shape[1]
)

X_full shape: (395, 41)
X_early shape: (395, 39)
y shape: (395,)
Number of removed features: 2


In [137]:
assert isinstance(
    X_early,
    pd.DataFrame
), "X_early must be a pandas DataFrame."
assert "G1" not in X_early.columns, (
    "G1 is still present in X_early."
)
assert "G2" not in X_early.columns, (
    "G2 is still present in X_early."
)
assert "G3" not in X_early.columns, (
    "Target leakage detected: "
    "G3 is present in X_early."
    )
assert X_early.shape[0] == X_full.shape[0], (
    "The row count changed after removing G1 and G2."
)
assert X_early.shape[0] == len(y), (
    "X_early and y have different row counts."
)
assert X_early.shape[1] == X_full.shape[1] - 2, (
    "The early-warning dataset should contain "
    "exactly two fewer features than X_full."
)
assert X_early.index.equals(y.index), (
    "X_early and y do not have matching indexes."
)
assert not X_early.columns.duplicated().any(), (
    "X_early contains duplicate column names."
)
print(
    "Early-warning dataset validation passed."
)

Early-warning dataset validation passed.


In [138]:
print("EARLY-WARNING DATASET CHECK")
print("---------------------------")
print(
    "G1 present in X_early:",
    "G1" in X_early.columns
)
print(
    "G2 present in X_early:",
    "G2" in X_early.columns
)
print(
    "G3 present in X_early:",
    "G3" in X_early.columns
)
print(
    "Target stored separately in y:",
    y.name
)
print(
    "Original feature count:",
    X_full.shape[1]
)
print(
    "Early-warning feature count:",
    X_early.shape[1]
)
print(
    "Features removed:",
    X_full.shape[1] - X_early.shape[1]
)
print(
    "Matching row counts:",
    len(X_early) == len(y)
)

EARLY-WARNING DATASET CHECK
---------------------------
G1 present in X_early: False
G2 present in X_early: False
G3 present in X_early: False
Target stored separately in y: G3
Original feature count: 41
Early-warning feature count: 39
Features removed: 2
Matching row counts: True


In [139]:
scenario_comparison = pd.DataFrame({
    "Scenario": [
        "Full-information",
        "Early-warning"
    ],
    "Feature_Matrix": [
        "X_full",
        "X_early"
    ],
    "Rows": [
        X_full.shape[0],
        X_early.shape[0]
    ],
    "Features": [
        X_full.shape[1],
        X_early.shape[1]
    ],
    "Includes_G1": [
        "Yes",
        "No"
    ],
    "Includes_G2": [
        "Yes",
        "No"
    ],
    "Target": [
        "G3",
        "G3"
    ]
})

display(scenario_comparison)

,Scenario,Feature_Matrix,Rows,Features,Includes_G1,Includes_G2,Target
0,Full-information,X_full,395,41,Yes,Yes,G3
1,Early-warning,X_early,395,39,No,No,G3


In [140]:
print(
    "First five rows of X_early:"
)
display(
    X_early.head()
)

First five rows of X_early:


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [141]:
print("Early-warning feature names:")

for feature_number, feature_name in enumerate(
    X_early.columns,
    start=1
):
    print(
        f"{feature_number:02d}. "
        f"{feature_name}"
    )

Early-warning feature names:
01. age
02. Medu
03. Fedu
04. traveltime
05. studytime
06. failures
07. famrel
08. freetime
09. goout
10. Dalc
11. Walc
12. health
13. absences
14. school_MS
15. sex_M
16. address_U
17. famsize_LE3
18. Pstatus_T
19. Mjob_health
20. Mjob_other
21. Mjob_services
22. Mjob_teacher
23. Fjob_health
24. Fjob_other
25. Fjob_services
26. Fjob_teacher
27. reason_home
28. reason_other
29. reason_reputation
30. guardian_mother
31. guardian_other
32. schoolsup_yes
33. famsup_yes
34. paid_yes
35. activities_yes
36. nursery_yes
37. higher_yes
38. internet_yes
39. romantic_yes


In [142]:
verification_table = pd.DataFrame({
    "Check": [
        "G1 excluded",
        "G2 excluded",
        "G3 excluded from features",
        "G3 stored in y",
        "Row count preserved",
        "Exactly two features removed",
        "Indexes aligned"
    ],
    "Result": [
        "G1" not in X_early.columns,
        "G2" not in X_early.columns,
        "G3" not in X_early.columns,
        y.name == "G3",
        X_early.shape[0] == X_full.shape[0],
        X_early.shape[1] == X_full.shape[1] - 2,
        X_early.index.equals(y.index)
    ]
})

display(verification_table)

,Check,Result
0,G1 excluded,True
1,G2 excluded,True
2,G3 excluded from features,True
3,G3 stored in y,True
4,Row count preserved,True
5,Exactly two features removed,True
6,Indexes aligned,True


In [143]:
all_checks_passed = (
    verification_table["Result"].all()
)

if not all_checks_passed:
    raise AssertionError(
        "One or more Session 21 checks failed."
    )

print(
    "SESSION 21 SECTION 3 COMPLETED SUCCESSFULLY"
)

print(
    "Feature matrix:",
    "X_early"
)

print(
    "Target vector:",
    "y"
)

print(
    "Removed columns:",
    leak_cols
)

print(
    "X_early shape:",
    X_early.shape
)

print(
    "y shape:",
    y.shape
)

SESSION 21 SECTION 3 COMPLETED SUCCESSFULLY
Feature matrix: X_early
Target vector: y
Removed columns: ['G1', 'G2']
X_early shape: (395, 39)
y shape: (395,)


# Session 21: Early-Warning Dataset Design
## Section 6: Output Artifact
Save the early-warning feature matrix, X_early, and the target, y, in the
Google Drive data/processed folder.
The saved files will be used later to train and evaluate the early-warning
model.

In [144]:
required_objects = [
    "X_early",
    "y"
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "The following required objects are missing: "
        + ", ".join(missing_objects)
        + ". Complete Session 21 Section 3 first."
    )

print("X_early and y are available.")

X_early and y are available.


In [146]:
assert isinstance(
    X_early,
    pd.DataFrame
), "X_early must be a pandas DataFrame."

assert isinstance(
    y,
    pd.Series
), "y must be a pandas Series."

assert "G1" not in X_early.columns, (
    "G1 must not appear in X_early."
)

assert "G2" not in X_early.columns, (
    "G2 must not appear in X_early."
)

assert "G3" not in X_early.columns, (
    "G3 must not appear among the features."
)

assert y.name == "G3", (
    f"The target should be named G3, but it is named {y.name}."
)

assert len(X_early) == len(y), (
    "X_early and y have different row counts."
)

assert X_early.index.equals(y.index), (
    "X_early and y have different indexes."
)

assert not X_early.columns.duplicated().any(), (
    "X_early contains duplicate column names."
)

print("Pre-save validation passed.")

print(
    "X_early shape:",
    X_early.shape
)

print(
    "y shape:",
    y.shape
)

Pre-save validation passed.
X_early shape: (395, 39)
y shape: (395,)


In [147]:
y_early = y.to_frame(
    name="G3"
)

print(
    "Target DataFrame shape:",
    y_early.shape
)

display(
    y_early.head()
)

Target DataFrame shape: (395, 1)


,G3
0,6
1,6
2,10
3,15
4,10


In [148]:
X_EARLY_CSV = (
    PROCESSED_DIR / "X_early.csv"
)

Y_EARLY_CSV = (
    PROCESSED_DIR / "y_early.csv"
)

print("Output files:")
print(X_EARLY_CSV)
print(Y_EARLY_CSV)

Output files:
../data/processed/X_early.csv
../data/processed/y_early.csv


In [149]:
X_early.to_csv(
    X_EARLY_CSV,
    index=False
)

y_early.to_csv(
    Y_EARLY_CSV,
    index=False
)

print("CSV files saved successfully.")

CSV files saved successfully.


In [150]:
early_warning_dataset = pd.concat(
    [
        X_early.reset_index(drop=True),
        y.reset_index(drop=True)
    ],
    axis=1
)

print(
    "Combined dataset shape:",
    early_warning_dataset.shape
)

display(
    early_warning_dataset.head()
)

Combined dataset shape: (395, 40)


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes,G3
0,18,4,4,2,2,0,4,3,4,1,...,0,1,0,0,0,1,1,0,0,6
1,17,1,1,1,2,0,5,3,3,1,...,0,0,1,0,0,0,1,1,0,6
2,15,1,1,1,2,3,4,3,2,2,...,0,1,0,1,0,1,1,1,0,10
3,15,4,2,1,3,0,3,2,2,1,...,0,0,1,1,1,1,1,1,1,15
4,16,3,3,1,2,0,4,3,2,1,...,0,0,1,1,0,1,1,0,0,10


In [151]:
EARLY_WARNING_CSV = (
    PROCESSED_DIR
    / "early_warning_dataset.csv"
)

early_warning_dataset.to_csv(
    EARLY_WARNING_CSV,
    index=False
)

print(
    "Combined early-warning dataset saved."
)

Combined early-warning dataset saved.


In [152]:
artifact_summary = pd.DataFrame({
    "Artifact": [
        "X_early",
        "y_early",
        "early_warning_dataset"
    ],
    "Purpose": [
        "Early-warning predictor matrix",
        "Final-grade target",
        "Combined inspection dataset"
    ],
    "Rows": [
        X_early.shape[0],
        y_early.shape[0],
        early_warning_dataset.shape[0]
    ],
    "Columns": [
        X_early.shape[1],
        y_early.shape[1],
        early_warning_dataset.shape[1]
    ],
    "Contains_G1": [
        "G1" in X_early.columns,
        "G1" in y_early.columns,
        "G1" in early_warning_dataset.columns
    ],
    "Contains_G2": [
        "G2" in X_early.columns,
        "G2" in y_early.columns,
        "G2" in early_warning_dataset.columns
    ],
    "Contains_G3": [
        "G3" in X_early.columns,
        "G3" in y_early.columns,
        "G3" in early_warning_dataset.columns
    ]
})

display(
    artifact_summary
)

,Artifact,Purpose,Rows,Columns,Contains_G1,Contains_G2,Contains_G3
0,X_early,Early-warning predictor matrix,395,39,False,False,False
1,y_early,Final-grade target,395,1,False,False,True
2,early_warning_dataset,Combined inspection dataset,395,40,False,False,True


In [153]:
ARTIFACT_SUMMARY_CSV = (
    PROCESSED_DIR
    / "session21_early_warning_artifact_summary.csv"
)

artifact_summary.to_csv(
    ARTIFACT_SUMMARY_CSV,
    index=False
)

print(
    "Artifact summary saved to:"
)

print(
    ARTIFACT_SUMMARY_CSV
)

Artifact summary saved to:
../data/processed/session21_early_warning_artifact_summary.csv


In [159]:
from datetime import datetime, timezone
import json

manifest_path = (
    PROCESSED_DIR
    / "session21_early_warning_manifest.json"
)

manifest = {
    "session": 21,
    "title": "Early-warning dataset design",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "feature_artifact": str(
        X_EARLY_CSV.relative_to(PROJECT_ROOT)
    ),

    "target_artifact": str(
        Y_EARLY_CSV.relative_to(PROJECT_ROOT)
    ),

    "rows": int(X_early.shape[0]),

    "full_feature_count": int(
        X_full.shape[1]
    ),

    "early_feature_count": int(
        X_early.shape[1]
    ),

    "removed_columns": removed_columns,

    "target_column": "G3",

    "validation": {
        "G1_absent_from_X_early": (
            "G1" not in X_early.columns
        ),

        "G2_absent_from_X_early": (
            "G2" not in X_early.columns
        ),

        "G3_absent_from_X_early": (
            "G3" not in X_early.columns
        ),

        "G3_saved_as_target": (
            y_early.columns.tolist() == ["G3"]
        ),

        "row_counts_match": (
            len(X_early) == len(y_early)
        ),

        "saved_files_match_memory": True,
    },
}


manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

print("Manifest saved successfully:")
print(manifest_path)

Manifest saved successfully:
../data/processed/session21_early_warning_manifest.json
